In [84]:
import sys
sys.path.append("..")

In [85]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [86]:
def get_model_adv_pga(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(np.stack(X_0)).float()
    X_r = torch.tensor(np.stack(X_r)).float()
    
    loss_fn = torch.nn.BCELoss(reduction='mean')
    cfr_adv = deepcopy(cfr)
    optimizer = optim.Adam(cfr_adv.parameters(), maximize=True)
    weights_min = [cfr.fc1.weight.data-alpha, cfr.fc2.weight.data-alpha, cfr.fc3.weight.data-alpha, cfr.out.weight.data-alpha]
    weights_max = [cfr.fc1.weight.data+alpha, cfr.fc2.weight.data+alpha, cfr.fc3.weight.data+alpha, cfr.out.weight.data+alpha]
    bias_min = [cfr.fc1.bias.data-alpha, cfr.fc2.bias.data-alpha, cfr.fc3.bias.data-alpha, cfr.out.bias.data-alpha]
    bias_max = [cfr.fc1.bias.data+alpha, cfr.fc2.bias.data+alpha, cfr.fc3.bias.data+alpha, cfr.out.bias.data+alpha]
        
    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    # while loss_diff > 1e-4:
    for epoch in range(pga_max_iter):
        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv(X_r)
        y_target = torch.ones(f_x.shape).float()
        bce_loss = loss_fn(f_x, y_target)
        cost = torch.dist(X_r, X_0, 1)
        loss = bce_loss + lamb*cost
        
        loss.backward()
        optimizer.step()
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
        
        # clamp model parameters to -alpha, alpha range
        cfr_adv.fc1.weight.data = cfr_adv.fc1.weight.data.clamp(weights_min[0], weights_max[0])
        cfr_adv.fc2.weight.data = cfr_adv.fc2.weight.data.clamp(weights_min[1], weights_max[1])
        cfr_adv.fc3.weight.data = cfr_adv.fc3.weight.data.clamp(weights_min[2], weights_max[2])
        cfr_adv.out.weight.data = cfr_adv.out.weight.data.clamp(weights_min[3], weights_max[3])
        
        cfr_adv.fc1.bias.data = cfr_adv.fc1.bias.data.clamp(bias_min[0], bias_max[0])
        cfr_adv.fc2.bias.data = cfr_adv.fc2.bias.data.clamp(bias_min[1], bias_max[1])
        cfr_adv.fc3.bias.data = cfr_adv.fc3.bias.data.clamp(bias_min[2], bias_max[2])
        cfr_adv.out.bias.data = cfr_adv.out.bias.data.clamp(bias_min[3], bias_max[3])
    
    wnorms = [
        torch.dist(cfr.fc1.weight.data, cfr_adv.fc1.weight.data, torch.inf),
        torch.dist(cfr.fc2.weight.data, cfr_adv.fc2.weight.data, torch.inf),
        torch.dist(cfr.fc3.weight.data, cfr_adv.fc3.weight.data, torch.inf),
        torch.dist(cfr.out.weight.data, cfr_adv.out.weight.data, torch.inf),
    ]
    
    bnorms = [
        torch.dist(cfr.fc1.bias.data, cfr_adv.fc1.bias.data, torch.inf),
        torch.dist(cfr.fc2.bias.data, cfr_adv.fc2.bias.data, torch.inf),
        torch.dist(cfr.fc3.bias.data, cfr_adv.fc3.bias.data, torch.inf),
        torch.dist(cfr.out.bias.data, cfr_adv.out.bias.data, torch.inf),
    ]
    
    # print(f'Final Loss: {loss}')
    # print(f'Num Iterations: {i}')
    # print(f'weights_alpha, bias_alpha: {max(wnorms), max(bnorms)}')
            
    return cfr_adv

In [87]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [88]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    
    theta_adv = deepcopy(theta_0)
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [89]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

In [90]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]

    if alpha != 0:
        if theta_adv_method=='L-1':
            theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
        else:
            theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = theta_0.copy()
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [91]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [92]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        t_0 = theta_0[i]
        w_0, b_0 = t_0[:-1], t_0[[-1]]
        if alpha != 0:
            w_0_adv, b_0_adv = calTheta(x_r, w_0, b_0, alpha, theta_adv_method)
        else:
            w_0_adv, b_0_adv = w_0.copy(), b_0.copy()

        clf.model.coef_ = w_0.reshape(1,-1)
        clf.model.intercept_ = b_0
        clf_adv.model.coef_ = w_0_adv.reshape(1,-1)
        clf_adv.model.intercept_ = b_0_adv

        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, w_0_adv, b_0_adv, True)

        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, None, None)

In [ ]:
# def runCostValidityTradeoff (results: dict, params: dict):
#     for algorithm in params['algorithms']:
#         for seed in params['seeds']:
#             # for v_alpha in params['alphas']:
#             for v_alpha in params['alphas'][algorithm]:
#                 # for v_lamb in params['lambdas']:
#                 for v_lamb in params['lambdas'][algorithm]:
#                     data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
#                     alpha = data["alpha"].unique().item()
#                     lamb = data["lambda"].unique().item()
#                     X_0 = np.stack(data["x_0"])
#                     X_r = np.stack(data["x_r"])
#                     # theta_0 = data["theta_0"].iloc[0]
#                     theta_0 = np.stack(data["theta_0"])

#                     if params['include_mask'] and algorithm != "L1PSD":
#                         data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
#                         mask_i = data_l1psd["i"].to_numpy()
#                         X_0 = X_0[mask_i]
#                         X_r = X_r[mask_i]
#                         theta_0 = theta_0[mask_i]
                            
#                     match params['adv_method']:
#                         case "ONE":
#                             res = evaluate_performance(X_0, X_r, theta_0[0], alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                         case "MANY":
#                             # theta_0 = np.stack(data["theta_0"].to_numpy(), axis=0)
#                             # if algorithm != "L1PSD":
#                             #     data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
#                             #     mask_i = data_l1psd["i"].to_numpy()
#                             #     X_0 = X_0[mask_i]
#                             #     X_r = X_r[mask_i]
#                             #     theta_0 = theta_0[mask_i]

#                             res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                         case "LARGESTALPHA":
#                             res = evaluate_performance(X_0, X_r, theta_0[0], max(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                         case "SMALLESTALPHA":
#                             res = evaluate_performance(X_0, X_r, theta_0[0], min(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                         case "THETA0":
#                             res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                         case "_":
#                             print(f"{params['adv_method']} does not exist!")

#                     results['algorithm'].append(algorithm)
#                     results['seed'].append(seed)
#                     results['alpha'].append(alpha)
#                     results['lambda'].append(lamb)
#                     results['Cost'].append(res['cost'])
#                     results['Current Validity'].append(res['m1_probability'])
#                     results['Worst Case Validity'].append(res['wc_probability'])
#                     # results['Worst Case Validity'].append(res['loss'])
#                     results['BCE Loss'].append(res['loss'])
#                     results['J'].append(res['J'])


#     # if params['include_base_model']:
#     #     for algo in ["BaseLineLInf", "BaseLineL1"]:
#     #         algorithm = params['algorithms'][0]
#     #         for seed in params['seeds']:
#     #             for v_alpha in params['alphas']:
#     #                 v_lamb = params['lambdas'][0]
#     #                 data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")

#     #                 alpha = data["alpha"].unique().item()
#     #                 lamb = 0
#     #                 theta_0 = data["theta_0"].iloc[0]
#     #                 X_0 = np.stack(data["x_0"])
#     #                 X_r = np.stack(data["x_0"])

#     #                 match params['adv_method']:
#     #                     case "ONE":
#     #                         res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
#     #                     case "MANY":
#     #                         res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
#     #                     case "LARGESTALPHA":
#     #                         res = evaluate_performance(X_0, X_r, theta_0, max(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
#     #                     case "SMALLESTALPHA":
#     #                         res = evaluate_performance(X_0, X_r, theta_0, min(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
#     #                     case "THETA0":
#     #                         res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
#     #                     case "_":
#     #                         print(f"{params['adv_method']} does not exist!")
                            
#     #                 results['algorithm'].append(algo)
#     #                 results['seed'].append(seed)
#     #                 results['alpha'].append(alpha)
#     #                 results['lambda'].append(lamb)
#     #                 results['Cost'].append(res['cost'])
#     #                 results['Current Validity'].append(res['m1_probability'])
#     #                 results['Worst Case Validity'].append(res['wc_probability'])
#     #                 results['BCE Loss'].append(res['loss'])
#     #                 results['J'].append(res['J'])

    
#     df_results = pd.DataFrame(results)
#     return df_results

In [144]:
def runCostValidityTradeoff (results: dict, params: dict):
    for algorithm in params['algorithms']:
        for seed in params['seeds']:
            for v_alpha in params['a_l'][algorithm].keys():
                # for v_lamb in params['lambdas']:
                for v_lamb in params['a_l'][algorithm][v_alpha]:
                    data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                    alpha = data["alpha"].unique().item()
                    lamb = data["lambda"].unique().item()
                    X_0 = np.stack(data["x_0"])
                    X_r = np.stack(data["x_r"])
                    # theta_0 = data["theta_0"].iloc[0]
                    theta_0 = np.stack(data["theta_0"])

                    if params['include_mask'] and algorithm != "L1PSD":
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                        mask_i = data_l1psd["i"].to_numpy()
                        X_0 = X_0[mask_i]
                        X_r = X_r[mask_i]
                        theta_0 = theta_0[mask_i]
                            
                    match params['adv_method']:
                        case "ONE":
                            res = evaluate_performance(X_0, X_r, theta_0[0], alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "MANY":
                            # theta_0 = np.stack(data["theta_0"].to_numpy(), axis=0)
                            # if algorithm != "L1PSD":
                            #     data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                            #     mask_i = data_l1psd["i"].to_numpy()
                            #     X_0 = X_0[mask_i]
                            #     X_r = X_r[mask_i]
                            #     theta_0 = theta_0[mask_i]

                            res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "LARGESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0[0], max(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "SMALLESTALPHA":
                            res = evaluate_performance(X_0, X_r, theta_0[0], min(params['alphas']), lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "THETA0":
                            res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                        case "_":
                            print(f"{params['adv_method']} does not exist!")

                    results['algorithm'].append(algorithm)
                    results['seed'].append(seed)
                    results['alpha'].append(alpha)
                    results['lambda'].append(lamb)
                    results['Cost'].append(res['cost'])
                    results['Current Validity'].append(res['m1_probability'])
                    results['Worst Case Validity'].append(res['wc_probability'])
                    results['BCE Loss'].append(res['loss'])
                    results['J'].append(res['J'])


    # if params['include_base_model']:
    #     for algo in ["BaseLineLInf", "BaseLineL1"]:
    #         algorithm = params['algorithms'][0]
    #         for seed in params['seeds']:
    #             for v_alpha in params['alphas']:
    #                 v_lamb = params['lambdas'][0]
    #                 data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")

    #                 alpha = data["alpha"].unique().item()
    #                 lamb = 0
    #                 theta_0 = data["theta_0"].iloc[0]
    #                 X_0 = np.stack(data["x_0"])
    #                 X_r = np.stack(data["x_0"])

    #                 match params['adv_method']:
    #                     case "ONE":
    #                         res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
    #                     case "MANY":
    #                         res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
    #                     case "LARGESTALPHA":
    #                         res = evaluate_performance(X_0, X_r, theta_0, max(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
    #                     case "SMALLESTALPHA":
    #                         res = evaluate_performance(X_0, X_r, theta_0, min(params['alphas']), lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
    #                     case "THETA0":
    #                         res = evaluate_performance_each(X_0, X_r, theta_0, 0, lamb, seed, algo, theta_adv_method="L-1" if "L1" in algo else "L-inf")
    #                     case "_":
    #                         print(f"{params['adv_method']} does not exist!")
                            
    #                 results['algorithm'].append(algo)
    #                 results['seed'].append(seed)
    #                 results['alpha'].append(alpha)
    #                 results['lambda'].append(lamb)
    #                 results['Cost'].append(res['cost'])
    #                 results['Current Validity'].append(res['m1_probability'])
    #                 results['Worst Case Validity'].append(res['wc_probability'])
    #                 results['BCE Loss'].append(res['loss'])
    #                 results['J'].append(res['J'])

    
    df_results = pd.DataFrame(results)
    return df_results

In [147]:
doAll = False

params = {}
# 'lr', 'nn'
params['base_model'] = 'lr'
# 'synthetic', 'german', 'sba'
params['data'] = 'german'
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# 'ONE', 'MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0'
params['adv_method'] = 'MANY'
params['include_base_model'] = False
params['include_mask'] = True


params['a_l'] = {params['algorithms'][0] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
                                                0.2: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5)},
            params['algorithms'][1] : {}, 
            params['algorithms'][2] : {}, 
            params['algorithms'][3] : {}}

# params['alphas'] = {params['algorithms'][0] : [0.1, 0.2],
#             params['algorithms'][1] : [0.1], 
#             params['algorithms'][2] : [0.1], 
#             params['algorithms'][3] : [0.1]}

# params['lambdas'] = {params['algorithms'][0] : {0.1: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
#                                                 0.2: np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5)},
#             params['algorithms'][1] : [], 
#             params['algorithms'][2] : [], 
#             params['algorithms'][3] : []}

# lr_german
# params['lambdas'] = {params['algorithms'][0] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
#             params['algorithms'][1] : [0.5,0.3,0.1,0.04,0.01,0.004,0.001], 
#             params['algorithms'][2] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7), 
#             params['algorithms'][3] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7)}

# nn_german
# params['lambdas'] = {params['algorithms'][0] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0, 4.0, 5.0]))).round(5),
#             params['algorithms'][1] : [3.0, 0.7, 0.3, 0.1, 0.05, 0.01, 0.001], 
#             params['algorithms'][2] : np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9), 
#             params['algorithms'][3] : np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9)}

# lr_sba
# params['lambdas'] = {params['algorithms'][0] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.3, 2.6, 2.8, 3.0, 3.1, 3.3, 3.5],
#             params['algorithms'][1] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.0, 3.5], 
#             params['algorithms'][2] : [0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5], 
#             params['algorithms'][3] : [0.08,0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5]}

# nn_sba
# params['lambdas'] = {params['algorithms'][0] :[0.00001, 0.0001, 0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5],
#             params['algorithms'][1] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5], 
#             params['algorithms'][2] : [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5], 
#             params['algorithms'][3] : [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5]}



results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}

df_results_list = []
if doAll:
    adv_methods = ['MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0']
    for adv_method in adv_methods:
        params['adv_method'] = adv_method
        df_results_list.append(runCostValidityTradeoff(results, params))
else:
    df_results = runCostValidityTradeoff(results, params)

[Alg1] [ seed=0 ] [ α=0.1 ] [ λ=0.001 ]: 100%|██████████| 16/16 [00:00<00:00, 2005.46it/s]


[Alg1] [ seed=4 ] [ α=0.2 ] [ λ=0.5 ]: 100%|██████████| 14/14 [00:00<00:00, 2302.12it/s]


In [124]:
print(f'{params["data"]}  |  {params["base_model"].upper()}')
df_results_avg = df_results.groupby(['algorithm', 'lambda'], as_index=False).mean(True)
df_results_im = df_results_avg.copy()
df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']] = df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str) + '±' + df_results.groupby(['algorithm', 'lambda'], as_index=False).std(numeric_only=True)[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str)

df_results_im

german  |  LR


,algorithm,lambda,seed,alpha,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.001,2.0,0.15,24.16±6.8,1.0±0.0,1.0±0.0,0.004121,0.03±0.01
1,Alg1,0.002,2.0,0.15,21.29±5.63,1.0±0.0,0.99±0.0,0.008262,0.05±0.01
2,Alg1,0.003,2.0,0.15,19.61±4.94,1.0±0.0,0.99±0.01,0.012423,0.07±0.02
3,Alg1,0.004,2.0,0.15,18.4±4.45,1.0±0.0,0.98±0.01,0.016604,0.09±0.02
4,Alg1,0.005,2.0,0.15,17.47±4.06,1.0±0.0,0.98±0.01,0.020805,0.11±0.03
5,Alg1,0.006,2.0,0.15,16.7±3.75,1.0±0.0,0.98±0.01,0.025027,0.13±0.03
6,Alg1,0.007,2.0,0.15,16.04±3.48,1.0±0.0,0.97±0.01,0.029270,0.14±0.04
7,Alg1,0.008,2.0,0.15,15.47±3.24,1.0±0.0,0.97±0.01,0.033533,0.16±0.04
8,Alg1,0.009,2.0,0.15,14.97±3.04,1.0±0.0,0.96±0.02,0.037818,0.17±0.04
9,Alg1,0.010,2.0,0.15,14.52±2.85,0.99±0.0,0.96±0.02,0.042125,0.19±0.05


In [125]:
if doAll:
    df_results_avg_list = []
    for df_results in df_results_list:
        df_results_avg_list.append(df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean())
else:
    df_results_avg = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()

In [126]:
# df_results_avg = df_results.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()

# px.line(df_results_avg, x='lambda', y='J', markers=True)
df_results_avg

,algorithm,lambda,alpha,seed,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.001,0.1,2.0,20.271633,0.999630,0.996757,0.003249,0.023520
1,Alg1,0.001,0.2,2.0,28.054656,0.999961,0.995020,0.004994,0.033049
2,Alg1,0.002,0.1,2.0,18.012547,0.999071,0.993514,0.006509,0.042534
3,Alg1,0.002,0.2,2.0,24.575012,0.999859,0.990040,0.010016,0.059166
4,Alg1,0.003,0.1,2.0,16.686437,0.998405,0.990270,0.009779,0.059839
5,Alg1,0.003,0.2,2.0,22.527670,0.999694,0.985060,0.015067,0.082650
6,Alg1,0.004,0.1,2.0,15.742285,0.997656,0.987027,0.013062,0.076031
7,Alg1,0.004,0.2,2.0,21.066648,0.999467,0.980081,0.020147,0.104413
8,Alg1,0.005,0.1,2.0,15.007418,0.996836,0.983784,0.016355,0.091392
9,Alg1,0.005,0.2,2.0,19.926775,0.999173,0.975101,0.025256,0.124890


In [127]:
# custom_colors = {
#     "LInf(Lamb = 0.1)": "#33FFFF",
#     "L1PSD(Lamb = 0.1)": "#FF3333",
#     "ROARL1(Lamb = 0.1)": "#33FF33",
#     "ROARLInf(Lamb = 0.1)": "#FF33FF",
#     "LInf(Lamb = 0.3)": "#33FFFF",
#     "L1PSD(Lamb = 0.3)": "#FF3333",
#     "ROARL1(Lamb = 0.3)": "#33FF33",
#     "ROARLInf(Lamb = 0.3)": "#FF33FF"
# }

# custom_colors = {
#     "LInf(Lamb = 0.1)": "#33FFFF",
#     "L1PSD(Lamb = 0.1)": "#FF3333",
#     "ROARL1(Lamb = 0.1)": "#33FF33",
#     "ROARLInf(Lamb = 0.1)": "#FF33FF",
#     "LInf(Lamb = 0.2)": "#33FFFF",
#     "L1PSD(Lamb = 0.2)": "#FF3333",
#     "ROARL1(Lamb = 0.2)": "#33FF33",
#     "ROARLInf(Lamb = 0.2)": "#FF33FF"
# }

data_map = {'synthetic': 'Synthetic', 'sba': 'Small Business Administration', 'german': 'German', 'income': 'ACS Income'}
model_map = {'lr': 'Logistic Regression', 'nn': 'Neural Network'}

# custom_colors = ["#33FFFF","#C8EFEF", "#FF3333", "#A71616", "#33FF33", "#207D20","#FF33FF", "#F7DAF7"]
custom_colors = ["#33FFFF", "#FF3333", "#33FF33","#FF33FF"]

# colors = ['#1f77b4', '#17becf', '#9467bd', '#e377c2', '#2ca02c'] # Synthesis
colors = ['#C7E8F0', "#7FCBDC", "#37AEC8", '#236F80', '#E2C2F4', "#BD74E7", "#9726D9", "#61188B"] # Synthesis
# colors = ['#17becf', '#e377c2', '#2ca02c'] # German
# colors = ['#17becf', '#9467bd', '#e377c2', '#2ca02c'] # SBA
nc = len(colors)

# fig = px.line(df_graph, 
#            x="Cost", y="Worst Case Validity", 
#            color="algorithm_lamb",
#            hover_data=["alpha", "lambda"],
#            title=f"{params['base_model']}_{params['data']}_{params['adv_method']}Adv",
#            markers=True,
#            color_discrete_map=custom_colors,
#            facet_col="lambda") 
# fig

In [128]:
def plotFigures(df_results_avg, params, should_plot_frontier=False):
    font_family = 'Times New Roman'
    font_color = 'black'
    width, height = 720, 540

    # symbols = ['x' for _ in range(len(params['lambdas']))] + ['circle']
    # size = [7 for _ in range(len(params['lambdas']))] + [5]

    fig = go.Figure()

    if should_plot_frontier:
        for i, alg in enumerate(params["algorithms"]):
            df_alg = df_results_avg.copy()
            df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
            x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
            df_alg = pd.DataFrame({'Algorithm': [f"{alg}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

            fig.add_trace(go.Scatter(
                x = df_alg['Cost'],
                y = df_alg['Worst Case Validity'],
                mode = 'lines+markers' if alg != 'wachter' else 'markers',
                name = f"{alg}",
                showlegend=True,
                # customdata=df_alg['alpha'],
                customdata=df_alg[['alpha', 'lambda']].to_numpy(),
                hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata[0]}<br>lambda: %{customdata[1]}'
            ))

        # if params['include_base_model']:
        #     for algo in ["BaseLineL1", "BaseLineLInf"]:
        #         df_alg = df_results_avg.copy()
        #         df_alg = df_alg[(df_alg['algorithm']==algo)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
        #         x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        #         df_alg = pd.DataFrame({'Algorithm': [f"{algo}" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask], 'lambda': df_alg['lambda'][mask]})

        #         baseline_y = df_alg['Worst Case Validity'].mean()
        #         fig.add_trace(go.Scatter(
        #             x=[0, df_results_avg['Cost'].max()],
        #             y=[baseline_y, baseline_y],
        #             mode="lines",
        #             line=dict(dash="dash"),
        #             name=f"{algo}",
        #             showlegend=True
        #         ))
    else:
        c = 0
        
        # for i, alg in enumerate(params["algorithms"]):
        #     for lamb in params['lambdas']:
        #         df_alg = df_results_avg.copy()
        #         df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['lambda']==lamb)]
        #         x, y = df_alg['Cost'], df_alg['Worst Case Validity']
        #         df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

        #         # df_alg = df_alg[(df_alg['algorithm']==alg)].sort_values(['Cost'], ascending=True).reset_index(drop=True)
        #         # x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        #         # df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask]})

        #         fig.add_trace(go.Scatter(
        #             x = df_alg['Cost'],
        #             y = df_alg['Worst Case Validity'],
        #             # marker = dict(color=colors[c], size=5),
        #             # marker = dict(color=custom_colors[c], size=5),
        #             mode = 'lines+markers' if alg != 'wachter' else 'markers',
        #             name = f"{alg} (λ={lamb})",
        #             showlegend=True,
        #             customdata=df_alg['alpha'],
        #             hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata}'
        #         ))
        #         c+=1
            
        for i, alg in enumerate(params["algorithms"]):
            # for alpha in params['alphas']:
            for alpha in params['alphas'][alg]:
                df_alg = df_results_avg.copy()
                df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['alpha']==alpha)]
                x, y = df_alg['Cost'], df_alg['Worst Case Validity']
                df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(alpha={alpha})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'lambda': df_alg['lambda']})

                fig.add_trace(go.Scatter(
                    x = df_alg['Cost'],
                    y = df_alg['Worst Case Validity'],
                    # marker = dict(color=colors[c], size=5),
                    # marker = dict(color=custom_colors[c], size=5),
                    mode = 'lines+markers' if alg != 'wachter' else 'markers',
                    name = f"{alg} (alpha={alpha})",
                    showlegend=True,
                    customdata=df_alg['lambda'],
                    hovertemplate='Cost: %{x}<br>Validity: %{y}<br>lambda: %{customdata}'
                ))
                c+=1


        # if params['include_base_model']:
        #     for algo in ["BaseLineL1", "BaseLineLInf"]:
        #         df_alg = df_results_avg.copy()
        #         df_alg = df_alg[(df_alg['algorithm']==algo) & (df_alg['lambda']==0)]
        #         x, y = df_alg['Cost'], df_alg['Worst Case Validity']
        #         df_alg = pd.DataFrame({'Algorithm': [f"{algo}_(λ=0)" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

        #         baseline_y = df_alg['Worst Case Validity'].mean()
        #         fig.add_trace(go.Scatter(
        #             x=[0, df_results_avg['Cost'].max()],
        #             y=[baseline_y, baseline_y],
        #             mode="lines",
        #             line=dict(dash="dash"),
        #             name=f"{algo}",
        #             showlegend=True
        #         ))

    fig.update_xaxes(
        title=dict(
            text='Cost',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            )
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_yaxes(
        title=dict(
            text='Worst Case Validity',
            font=dict(
                family=font_family,
                color=font_color,
                size=25
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )


    fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        title =dict(
            # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
            text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
            x= 0.5, 
            font=dict(family=font_family, size=20)
            ),
        legend=dict(
            x=0.975, 
            y=0.025, 
            orientation='v',
            xanchor='right',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            # range=[0,21], # german_lr
            # range=[0, 8], # german_nn
            range=[0,6], # sba
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            # range=[0.2,1.1], # german_lr
            # range=[0.3,1.1], # german_nn
            range=[-0.1,1.1], # sba
        )
    )

    return fig

In [129]:
should_plot_frontier = False
fig = plotFigures(df_results_avg, params, should_plot_frontier)

print(f'{params["data"]}  |  {params["base_model"].upper()}')
fig.show()

german  |  LR


In [82]:
# if should_plot_frontier:
#     figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_frontiers" 
#     fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-15\\" + 
#       figName + f".html")
# else:
#       figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_alpha0.1" 
#       fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-15\\" + 
#       figName + f".html")

In [ ]:
# if should_plot_frontier:
#     figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_frontiers" 
#     fig.write_image(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-15\\" + 
#       figName + f".pdf")
# else:
#       figName = f"{params['base_model']}_{params['data']}_{params['adv_method']}_alpha0.1" 
#       fig.write_image(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-15\\" + 
#       figName + f".pdf")# 